# parserInvestfunds — выгрузка ПИФов с pro.investfunds.ru **без фильтров**

Что поменялось против прежней редакции:

| было | стало |
|---|---|
| фильтры: тип (ОПИФ/БПИФ/ИПИФ) × категория × специализация, 6 книг | фильтров нет, отмечено всё → **один** набор файлов |
| периоды: полугодия, список дат руками | **кварталы** назад от `сегодня - DAYS_BACK` (X в хардкоде) |
| сшивка через Excel/xlwings + `strip_inner_headers` | скачанные файлы читаются по очереди в **один `DataFrame df`** |
| результат: `akcii_investfundsОПИФЫ.xlsx` и ещё 5 книг | результат: `df` в памяти (сохранение — последняя ячейка, по желанию) |

Excel на машине больше не нужен — вся сборка в pandas. Файлы падают в папку
рядом с кодом (`./investfunds_xlsx`), а не туда, где лежит venv.

**Плата за отказ от фильтров:** в выгрузке сайта нет колонок «тип фонда» и
«категория» — раньше их заменяло имя книги (`akcii_…ОПИФЫ`). В общем `df`
останутся только колонки сайта (Дата / Фонд / УК / Пай / СЧА); тип и категорию,
если понадобятся, придётся подтягивать отдельным справочником.

**Кодировка.** Кнопка CSV отдаёт файл в cp1251 (ANSI Windows), и `pd.read_csv`
с кодировкой по умолчанию падает на первом же байте слова «Дата»
(`'utf-8' codec can't decode byte 0xc4 in position 0`). Поэтому в ячейке 4
байты декодируются вручную (`decode_csv`: utf-8 → cp1251 → cp1251 с заменой),
а если сайт отдал уже испорченный текст («Äàòà» вместо «Дата»), его
переигрывает `fix_all_encoding`. Важно: эта функция срабатывает только на
кракозябрах (`is_mojibake`) — на правильной кириллице второй проход превратил
бы «Дата» в «????».

Порядок прогона: ячейка 1 (настройки) → 2 (утилиты) → 3 (скачивание, Selenium +
msedgedriver.exe) → 4 (сборка `df`) → 5 (сводка). Ячейку 4 можно крутить
отдельно, не перекачивая файлы. Ячейка 4b (одиночный файл → xlsx) нужна редко
и требует, чтобы ячейка 4 уже была прогнана.

In [219]:
# =============================================================================
# 1. Настройки и периоды
# =============================================================================
import datetime
import os
import re
import time
from pathlib import Path
import moex_cls as mcls
import os
import numpy as np
import pandas as pd
import pyodbc

PROXY = "http://smsk01gw79.vtbc.dmz:3128"
os.environ["HTTP_PROXY"] = PROXY
os.environ["HTTPS_PROXY"] = PROXY
# ------------------------------------------------------------------ хардкод ---
DAYS_BACK = 10        # X: правый край самого свежего периода = сегодня - X дней
N_QUARTERS = 22      # сколько кварталов тянуть назад (22 ~ с 3 кв. 2021)

LOGIN = "baxa"
PASSWORD = "mozart"



DOWNLOAD_TIMEOUT = 600   # сек на одну выгрузку: без фильтров файл крупный
SETTLE_AFTER_DATES = 2   # сек на пересборку таблицы под новый диапазон
RETRIES = 1              # повторов на период, если выгрузка не доехала
SKIP_EXISTING = True     # уже скачанный период повторно не тянуть
KEEP_SOURCE_COL = False  # оставить в df служебную колонку с именем файла

# --------------------------------------------------------------------- пути ---
# Папка с кодом, а не с venv. В Jupyter у ячейки нет __file__, поэтому берём cwd
# ядра: для ноутбука это его собственная папка. В .py-скрипте замените строку на
# BASE_DIR = Path(__file__).resolve().parent
BASE_DIR = Path.cwd()
DOWNLOAD_DIR = BASE_DIR / "investfunds_xlsx"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# msedgedriver.exe: сначала рядом с кодом, потом по известным местам.
# Selenium тут работает только с этим драйвером, поэтому путь ищем, а не угадываем.
DRIVER_CANDIDATES = (
    BASE_DIR / "msedgedriver.exe",
    Path(r"C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0\msedgedriver.exe"),
    Path.home() / "Downloads" / "msedgedriver.exe",
)


def resolve_driver(candidates=DRIVER_CANDIDATES) -> Path:
    """Первый существующий путь к msedgedriver.exe или понятная ошибка."""
    for p in candidates:
        if Path(p).is_file():
            return Path(p)
    raise FileNotFoundError(
        "msedgedriver.exe не найден. Положите его рядом с кодом "
        f"({BASE_DIR}) или допишите путь в DRIVER_CANDIDATES. Искал:\n  "
        + "\n  ".join(str(p) for p in candidates)
    )


# ------------------------------------------------------- кварталы от «сегодня»
SITE_FMT = "%d.%m.%Y"   # формат календаря на сайте


def _q_start(d: datetime.date) -> datetime.date:
    """Первое число квартала, в который попала дата."""
    return d.replace(month=3 * ((d.month - 1) // 3) + 1, day=1)


def quarter_periods(anchor: datetime.date, n: int):
    """
    n кварталов назад от anchor: [(начало, конец), ...], от свежего к старому.

    Самый свежий период обрезан по anchor — квартал ещё не закончился. Остальные
    полные: шаг «первое число квартала минус день» сам попадает на конец
    предыдущего квартала, так что длину квартала считать не нужно.
    """
    out, cur = [], anchor
    for _ in range(n):
        start = _q_start(cur)
        out.append((start, cur))
        cur = start - datetime.timedelta(days=1)
    return out


ANCHOR_DATE = datetime.date.today() - datetime.timedelta(days=DAYS_BACK)
PERIODS = quarter_periods(ANCHOR_DATE, N_QUARTERS)

print(f"сегодня {datetime.date.today():{SITE_FMT}}, якорь (-{DAYS_BACK} дн.) "
      f"{ANCHOR_DATE:{SITE_FMT}}, периодов {len(PERIODS)}")
print(f"файлы кладём в {DOWNLOAD_DIR}")
for s, e in PERIODS:
    print(f"  {s:{SITE_FMT}} - {e:{SITE_FMT}}")

сегодня 26.10.2026, якорь (-10 дн.) 16.10.2026, периодов 22
файлы кладём в c:\Users\tmp_SKosyrkov\Downloads\py_ex\peers_vs_funds_2\investfunds_xlsx
  01.10.2026 - 16.10.2026
  01.07.2026 - 30.09.2026
  01.04.2026 - 30.06.2026
  01.01.2026 - 31.03.2026
  01.10.2025 - 31.12.2025
  01.07.2025 - 30.09.2025
  01.04.2025 - 30.06.2025
  01.01.2025 - 31.03.2025
  01.10.2024 - 31.12.2024
  01.07.2024 - 30.09.2024
  01.04.2024 - 30.06.2024
  01.01.2024 - 31.03.2024
  01.10.2023 - 31.12.2023
  01.07.2023 - 30.09.2023
  01.04.2023 - 30.06.2023
  01.01.2023 - 31.03.2023
  01.10.2022 - 31.12.2022
  01.07.2022 - 30.09.2022
  01.04.2022 - 30.06.2022
  01.01.2022 - 31.03.2022
  01.10.2021 - 31.12.2021
  01.07.2021 - 30.09.2021


In [220]:
# =============================================================================
# 2. Утилиты файловой системы: ждём докачку, раскладываем файлы по периодам
# =============================================================================
PARTIAL_SUFFIXES = (".crdownload", ".tmp", ".partial")


def dir_snapshot(folder) -> set:
    return set(os.listdir(folder))


def wait_for_new_download(folder, before: set, timeout: int = DOWNLOAD_TIMEOUT) -> Path:
    """Ждёт появления нового полностью догруженного файла и возвращает его путь."""
    deadline = time.time() + timeout
    last_size = -1
    while time.time() < deadline:
        fresh = [f for f in os.listdir(folder)
                 if f not in before and not f.endswith(PARTIAL_SUFFIXES)]
        if fresh:
            path = max((Path(folder) / f for f in fresh),
                       key=lambda p: p.stat().st_mtime)
            size = path.stat().st_size
            if size > 0 and size == last_size:   # два замера подряд совпали
                return path
            last_size = size
        time.sleep(0.7)
    raise TimeoutError(f"Новый файл не появился в {folder} за {timeout} с")


def period_path(start, fin, suffix=".xlsx") -> Path:
    """
    Имя под период: investfunds_20260701_20260930.xlsx.

    Детерминированное имя вместо «Выгрузка (7).xls» даёт две вещи: сортировка
    по имени = сортировка по времени, и повторный прогон того же периода
    перезатирает файл, а не плодит копии.
    """
    return DOWNLOAD_DIR / f"investfunds_{start:%Y%m%d}_{fin:%Y%m%d}{suffix}"


def existing_period_file(start, fin):
    """Уже скачанный файл периода (любое расширение) или None."""
    hits = sorted(DOWNLOAD_DIR.glob(f"investfunds_{start:%Y%m%d}_{fin:%Y%m%d}.*"))
    return hits[0] if hits else None

def clear_filters(driver, wait, container_ids=("cat_check_list", "spec_check_list"),
                  settle=0.4, fast=True, attempts=2, strict=True):
    """
    Снимает все отмеченные чекбоксы в указанных контейнерах (на свежей странице
    отмечено всё). Один JS-проход на контейнер + добивание WebDriver-кликами.

    container_ids : строка или кортеж; порядок важен — см. замечание ниже
    fast          : снимать одним execute_script без round-trip'ов на каждый чекбокс
    attempts      : сколько раз повторить проход, если после перерисовки что-то вернулось
    strict        : RuntimeError, если снять всё так и не удалось

    Возвращает {container_id: сколько чекбоксов осталось отмеченными} (в норме нули).
    """
    if isinstance(container_ids, str):
        container_ids = (container_ids,)

    residual = {}
    for cid in container_ids:
        try:
            wait.until(lambda d, c=cid: d.find_elements(
                By.CSS_SELECTOR, f"#{c} input[type=checkbox]"))
        except TimeoutError:          # контейнера нет/пуст — например, spec после сброса категорий
            residual[cid] = 0
            continue

        for _ in range(attempts):
            if fast:
                driver.execute_script(
                    "document.querySelectorAll(arguments[0] + ' input[type=checkbox]')"
                    ".forEach(function (b) { if (b.checked) b.click(); });",
                    "#" + cid,
                )

In [221]:
# =============================================================================
# 3. Скачивание: Selenium + msedgedriver.exe, фильтры НЕ применяются
# =============================================================================
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)

# type_check_list  - тип фонда (ОПИФ/БПИФ/ИПИФ/ЗПИФ)
# cat_check_list   - категория (акции, облигации, ...)
# spec_check_list  - специализация внутри категории
FILTER_CONTAINERS = ("type_check_list", "cat_check_list", "spec_check_list")


def filter_state(driver, container_id):
    """(отмечено, всего) по контейнеру — чтобы глазами видеть, что фильтра нет."""
    boxes = driver.find_elements(
        By.CSS_SELECTOR, f"#{container_id} input[type=checkbox]")
    on = 0
    for b in boxes:
        try:
            on += bool(b.is_selected())
        except StaleElementReferenceException:
            pass
    return on, len(boxes)


def ensure_all_checked(driver, containers=FILTER_CONTAINERS, settle=0.6, attempts=2):
    """
    «Не фильтровать» = отмечено всё. На свежей странице так и есть, но профиль
    Edge (--user-data-dir) умеет помнить прошлый выбор, поэтому недостающие галки
    доставляем сами — иначе молча получим урезанную выгрузку.

    Порядок контейнеров важен: тип -> категории -> специализации, нижние списки
    перерисовываются обработчиком верхних. Клик делаем через JS .click(), а не
    присваиванием b.checked = true: присваивание не шлёт событий, и inline
    onclick=setspecs()/setuk() не выполнится.
    """
    report = {}
    for cid in containers:
        for _ in range(attempts):
            on, total = filter_state(driver, cid)
            if total and on == total:
                break
            driver.execute_script(
                "document.querySelectorAll(arguments[0] + ' input[type=checkbox]')"
                ".forEach(function (b) { if (!b.checked) b.click(); });",
                "#" + cid,
            )
            time.sleep(settle)
        report[cid] = filter_state(driver, cid)   # в норме (N, N) по каждому
    return report


DRIVER_PATH = resolve_driver()
print("драйвер:", DRIVER_PATH)

edge_options = webdriver.EdgeOptions()
# edge_options.add_argument("--headless=new")
edge_options.add_argument("--window-size=1920,1080")
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--use-system-proxy-resolver")
user_data_dir = os.path.expanduser("~") + r"\AppData\Local\Microsoft\User Data"
edge_options.add_argument(f"--user-data-dir={user_data_dir}")
edge_options.add_argument("--profile-directory=Default")
edge_options.add_experimental_option(
    "prefs",
    {
        # качаем рядом с кодом, а не в папку venv и не в системные «Загрузки»
        "download.default_directory": str(DOWNLOAD_DIR),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    },
)
os.environ.pop("HTTP_PROXY", None)
os.environ.pop("HTTPS_PROXY", None)
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)

driver = webdriver.Edge(
    options=edge_options,
    service=Service(executable_path=str(DRIVER_PATH)),
)
wait = WebDriverWait(driver, 20)

saved, failed = [], []
try:
    driver.get("https://pro.investfunds.ru/")
    driver.get("https://pro.investfunds.ru/quotes/")

    try:
        ln = driver.find_element(By.ID, "loginName")
        ln.click()
        ln.send_keys(LOGIN)
        pw = driver.find_element(By.ID, "loginPass")
        pw.click()
        pw.send_keys(PASSWORD)
        driver.find_element(
            By.CSS_SELECTOR, "input.header-user-login-submit[type='submit']"
        ).click()
        time.sleep(2)
    except NoSuchElementException:
        # профиль Edge помнит сессию — формы логина на странице просто нет
        print("форма логина не найдена, похоже уже залогинены")

    wait.until(EC.presence_of_element_located((By.ID, "csv")))
    print("фильтры (отмечено/всего):", ensure_all_checked(driver))
    
    clear_filters(driver, wait, container_ids=['type_check_list'])
    wait.until(EC.element_to_be_clickable((By.ID, "t1"))).click()
    wait.until(EC.element_to_be_clickable((By.ID, "t4"))).click()

    for i, (start, fin) in enumerate(PERIODS, 1):
        tag = f"[{i}/{len(PERIODS)}] {start:{SITE_FMT}}-{fin:{SITE_FMT}}"

        have = existing_period_file(start, fin)
        if SKIP_EXISTING and have:
            print(f"{tag}: уже лежит, пропуск ({have.name})")
            saved.append(have)
            continue

        for attempt in range(RETRIES + 1):
            try:
                st_dat = driver.find_element(By.ID, "calendar-1")
                fin_dat = driver.find_element(By.ID, "calendar-2")

                st_dat.send_keys(Keys.CONTROL + "a")
                st_dat.send_keys(Keys.BACKSPACE)
                st_dat.send_keys(f"{start:{SITE_FMT}}")

                fin_dat.send_keys(Keys.CONTROL + "a")
                fin_dat.send_keys(Keys.BACKSPACE)
                fin_dat.send_keys(f"{fin:{SITE_FMT}}")
                fin_dat.send_keys(Keys.ESCAPE)   # закрыть всплывающий календарь

                time.sleep(SETTLE_AFTER_DATES)   # дать таблице пересобраться

                before = dir_snapshot(DOWNLOAD_DIR)
                driver.find_element(By.ID, "csv").click()
                got = wait_for_new_download(DOWNLOAD_DIR, before)

                dst = period_path(start, fin, got.suffix.lower() or ".xlsx")
                os.replace(got, dst)
                saved.append(dst)
                print(f"{tag}: {dst.name}, {dst.stat().st_size / 1024:.0f} КБ")
                break
            except (TimeoutError, TimeoutException) as err:
                # один период не доехал — остальные всё равно забираем
                print(f"{tag}: попытка {attempt + 1} мимо — {err}")
                if attempt == RETRIES:
                    failed.append((f"{start:{SITE_FMT}}", f"{fin:{SITE_FMT}}", repr(err)))
finally:
    driver.quit()

print(f"\nскачано {len(saved)}, провалов {len(failed)}")
for f in failed:
    print("  провал:", f)

драйвер: C:\Users\tmp_SKosyrkov\Downloads\py_ex\general_0\msedgedriver.exe
фильтры (отмечено/всего): {'type_check_list': (4, 4), 'cat_check_list': (18, 18), 'spec_check_list': (88, 88)}
[1/22] 01.10.2026-16.10.2026: investfunds_20261001_20261016.csv, 329 КБ
[2/22] 01.07.2026-30.09.2026: investfunds_20260701_20260930.csv, 1742 КБ
[3/22] 01.04.2026-30.06.2026: investfunds_20260401_20260630.csv, 1557 КБ
[4/22] 01.01.2026-31.03.2026: investfunds_20260101_20260331.csv, 1349 КБ
[5/22] 01.10.2025-31.12.2025: investfunds_20251001_20251231.csv, 1542 КБ
[6/22] 01.07.2025-30.09.2025: investfunds_20250701_20250930.csv, 1520 КБ
[7/22] 01.04.2025-30.06.2025: investfunds_20250401_20250630.csv, 1319 КБ
[8/22] 01.01.2025-31.03.2025: investfunds_20250101_20250331.csv, 1258 КБ
[9/22] 01.10.2024-31.12.2024: investfunds_20241001_20241231.csv, 1383 КБ
[10/22] 01.07.2024-30.09.2024: investfunds_20240701_20240930.csv, 1371 КБ
[11/22] 01.04.2024-30.06.2024: investfunds_20240401_20240630.csv, 1232 КБ
[12/22] 01

In [222]:
# =============================================================================
# 4b. Старый путь «один файл -> одна книга» (необязательный)
#     Нужен, только если файл периода надо отдать в Excel отдельно. Прогонять
#     ПОСЛЕ ячейки 4: берёт её read_sheet / extract_block / typed, а значит и
#     ремонт кодировки (fix_all_encoding) — своей копии больше не держит.
#
#     Прежняя версия чинила кодировку безусловным df.map(fix_all_encoding).
#     Так делать нельзя: на правильной кириллице второй проход даёт '????'.
# =============================================================================
CLEAN_OUT_DIR = Path(r"C:/Users/tmp_SKosyrkov/Documents/files_for_peers_v_funds/test")


def clean_uploaded_df(path_to_that_csv, start_date, finish_date):
    """Файл периода -> xlsx без мусора."""
    part, _, _ = extract_block(read_sheet(path_to_that_csv), str(path_to_that_csv))
    part, date_col, _ = typed(part)

    part = part[part[date_col].notna()]                     # подвальные примечания
    part = part[part["УК"] != "ТКБ Инвестмент Партнерс"]
    part = part[part["СЧА"].notna()]                        # было "Не раскрывается"
    part = part.drop_duplicates()                           # повтор блока в хвосте файла

    CLEAN_OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = CLEAN_OUT_DIR / (f"funds_{start_date.replace('.', '')}"
                           f"_{finish_date.replace('.', '')}.xlsx")
    part.to_excel(out, index=False)
    return str(out)

In [223]:
# =============================================================================
# 4. Сборка одного DataFrame из скачанных файлов
#    (замена прежней связки xlwings + strip_inner_headers)
# =============================================================================
import csv
import io

HEADER_VALUES = ("Дата", "Фонд", "УК", "Пай", "СЧА")
_WANT = {h.lower() for h in HEADER_VALUES}
_NAME_RE = re.compile(r"investfunds_(\d{8})_(\d{8})")


def _norm(v) -> str:
    """Ячейка без nbsp, двойных пробелов и регистра — для сравнения со шапкой."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    return re.sub(r"\s+", " ", str(v).replace("\xa0", " ")).strip().lower()


# ----------------------------------------------------------- кодировка CSV ----
# Кнопка CSV на сайте отдаёт файл в cp1251 (ANSI Windows), поэтому pandas с
# кодировкой по умолчанию падает на первом же байте слова «Дата» (0xC4):
# UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc4 in position 0.
# Лечим в два слоя: правильно декодируем байты файла, а если сайт отдал уже
# испорченный текст (cp1251, прочитанный как latin1: «Äàòà» вместо «Дата»),
# переигрываем декодирование через fix_all_encoding.
CSV_SUFFIXES = (".csv", ".txt", "")
CSV_ENCODINGS = ("utf-8-sig", "cp1251")   # порядок важен: utf-8 строгая и отвалится сама
CSV_SEPS = (";", "\t", ",", "|")

_CYRILLIC = re.compile(r"[Ѐ-ӿ]")
# Кириллица cp1251 (байты A8, B8, C0-FF), прочитанная как latin1/cp1252, даёт
# символы из этого диапазона. Требуем серию из 2+ подряд: одинокое «é» в
# латинском названии фонда — не кракозябра.
_MOJIBAKE_RUN = re.compile(r"[¨¸À-ÿ]{2,}")


def is_mojibake(v) -> bool:
    """Строка похожа на cp1251, прочитанный чужой кодировкой."""
    if not isinstance(v, str):
        return False
    bad = sum(len(run) for run in _MOJIBAKE_RUN.findall(v))
    return bad > 0 and bad > 10 * len(_CYRILLIC.findall(v))


def fix_all_encoding(text):
    """
    Кракозябры -> читаемый текст: байты возвращаем как latin1 и перечитываем
    как cp1251.

    Проверка is_mojibake обязательна, а не для красоты: на правильной строке
    второй проход всё ломает — cp1252 не умеет кириллицу, 'Дата' превратится
    в '????'. Нестроки и уже читаемый текст отдаём как есть.
    """
    if not isinstance(text, str) or not is_mojibake(text):
        return text
    try:
        return text.encode("latin1").decode("cp1251")
    except UnicodeEncodeError:
        # упало на спецсимволах (тире, кавычки) — подменяем их через cp1252
        return text.encode("cp1252", errors="replace").decode("cp1251", errors="replace")
    except Exception:
        return text


def fix_frame_encoding(frame: pd.DataFrame, name: str = "") -> pd.DataFrame:
    """
    Тот же ремонт, но для готового кадра (xlsx/HTML, где байтов уже не видно).

    Решение принимаем один раз на файл по образцу из первых строк, а не на
    каждую ячейку: на 50 тыс. строк это заметная разница, а кодировка внутри
    одного файла не меняется.
    """
    sample = [v for v in frame.head(50).to_numpy().ravel() if isinstance(v, str)]
    if not any(is_mojibake(v) for v in sample):
        return frame
    print(f"{name}: текст в кракозябрах — чиню fix_all_encoding")
    return frame.apply(lambda col: col.map(fix_all_encoding))


def decode_csv(raw: bytes, name: str = "") -> str:
    """Байты файла -> текст: сначала честные кодировки, потом ремонт."""
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"):   # редкий, но встречающийся экспорт
        return raw.decode("utf-16")
    for enc in CSV_ENCODINGS:
        try:
            text = raw.decode(enc)
        except UnicodeDecodeError:
            continue
        if is_mojibake(text):
            print(f"{name}: {enc} дал кракозябры — чиню fix_all_encoding")
            text = fix_all_encoding(text)
        return text
    # ни одна не подошла: cp1251 с заменой битых байт, чтобы не потерять файл
    print(f"{name}: ни одна из {CSV_ENCODINGS} не подошла, читаю cp1251 с заменой")
    return raw.decode("cp1251", errors="replace")


def sniff_sep(text: str) -> str:
    """
    Разделитель по первым строкам. Считать голые запятые нельзя: в числах
    '1 234,56' их полно. Поэтому берём тот символ, который даёт самую широкую
    и при этом самую повторяемую ширину строки.
    """
    sample = text.splitlines()[:50]
    best, best_score = CSV_SEPS[0], 0
    for sep in CSV_SEPS:
        widths = [len(row) for row in csv.reader(sample, delimiter=sep)]
        if not widths:
            continue
        top = max(set(widths), key=widths.count)
        score = (top - 1) * widths.count(top)
        if top > 1 and score > best_score:
            best, best_score = sep, score
    return best


def csv_frame(text: str) -> pd.DataFrame:
    """
    Текст -> кадр без шапки. Читаем модулем csv, а не токенайзером pandas:
    в выгрузке строки рваные (заголовок периода сверху — одно поле на строку),
    а pandas на такой ширине падает «Expected 1 fields, saw 5». Короткие
    строки добиваем пустыми ячейками — их потом снимет extract_block.
    """
    rows = list(csv.reader(io.StringIO(text), delimiter=sniff_sep(text)))
    if not rows:
        return pd.DataFrame()
    width = max(len(r) for r in rows)
    return pd.DataFrame([r + [""] * (width - len(r)) for r in rows], dtype=object)


def read_sheet(path) -> pd.DataFrame:
    """
    Файл периода как есть, без шапки: CSV в cp1251, xlsx/xls или HTML-таблица
    под расширением .xls (кнопка XLS на сайте иногда отдаёт именно её, и на
    ней read_excel падает — выручает read_html).
    """
    path = Path(path)
    problems = []
    if path.suffix.lower() in CSV_SUFFIXES:
        try:
            text = decode_csv(path.read_bytes(), path.name)
            if text.lstrip()[:1] == "<":        # вместо CSV приехала HTML-страница
                raise ValueError("файл начинается с '<' — это HTML, не CSV")
            return fix_frame_encoding(csv_frame(text), path.name)
        except Exception as err_csv:
            problems.append(f"как CSV ({err_csv})")
    else:
        try:
            return fix_frame_encoding(
                pd.read_excel(path, header=None, dtype=object), path.name)
        except Exception as err_excel:
            problems.append(f"как Excel ({err_excel})")

    # Последний шанс: HTML-таблица под чужим расширением. Кодировку и здесь
    # берём на себя — read_html на cp1251-байтах с чужой кодировкой молча
    # подставит U+FFFD, а такой текст уже ничем не починить.
    try:
        # thousands=None: иначе read_html прочитает '721,39' как 72139
        tables = pd.read_html(
            io.StringIO(decode_csv(path.read_bytes(), path.name)), thousands=None)
    except Exception as err_html:
        tables = None
        problems.append(f"как HTML ({err_html})")
    if not tables:
        raise RuntimeError(f"{path.name}: не читается " + ", ни ".join(problems))
    t = max(tables, key=len)
    if not isinstance(t.columns, pd.RangeIndex):   # шапку забрал <th> — вернём строкой
        head = pd.DataFrame([list(t.columns)])
        t = pd.concat([head, t.set_axis(range(t.shape[1]), axis=1)], ignore_index=True)
    return fix_frame_encoding(t.set_axis(range(t.shape[1]), axis=1).astype(object), path.name)


def extract_block(df: pd.DataFrame, name: str):
    """
    То же, что прежние ANCHOR='C4' + Ctrl+A + strip_inner_headers, но в pandas:
    ищем строку-шапку, берём только её непустые столбцы (так отваливаются пустые
    A/B и заголовок выгрузки сверху), ниже выкидываем повторы шапки и пустые строки.

    Возвращает (кадр, убрано_шапок, убрано_пустых).
    """
    norm = df.apply(lambda col: col.map(_norm))

    hdr = None
    for i in range(len(norm)):
        # >=3 совпадений, а не все 5: страховка, если сайт переименует столбец
        if len(set(norm.iloc[i]) & _WANT) >= 3:
            hdr = i
            break
    if hdr is None:
        raise ValueError(
            f"{name}: не нашёл строку-шапку {HEADER_VALUES}. Если в первых "
            f"строках видно 'Äàòà' — кракозябра проехала мимо fix_all_encoding")
    cols = [j for j in range(df.shape[1]) if norm.iat[hdr, j]]
    head = [re.sub(r"\s+", " ", str(df.iat[hdr, j]).replace("\xa0", " ")).strip()
            for j in cols]

    body = df.iloc[hdr + 1:, cols]
    if body.empty:
        return pd.DataFrame(columns=head), 0, 0

    bn = norm.iloc[hdr + 1:, cols]
    key = bn.apply(lambda r: "|".join(x for x in r if x), axis=1)  # ключ строки без пустых
    hdr_key = "|".join(x for x in norm.iloc[hdr, cols] if x)

    is_hdr = key == hdr_key
    is_empty = key == ""
    out = body.loc[~(is_hdr | is_empty)].copy()
    out.columns = head
    return out.reset_index(drop=True), int(is_hdr.sum()), int(is_empty.sum())


def to_num(s: pd.Series) -> pd.Series:
    """'1 234,56' / 1234.56 -> float; прочерки и пустые -> NaN."""
    t = (s.astype(str)
          .str.replace("\xa0", "", regex=False)
          .str.replace(" ", "", regex=False)
          .str.replace(",", ".", regex=False))
    return pd.to_numeric(t, errors="coerce")


def to_date(s: pd.Series) -> pd.Series:
    """
    В столбце мешаются строки 'дд.мм.гггг' и настоящие datetime (их распознал
    openpyxl). Формат для строк задаём явно: pd.to_datetime('01.02.2026') без
    формата — это 2 января, а не 1 февраля.
    """
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    as_str = s.map(lambda v: isinstance(v, str))
    if as_str.any():
        out.loc[as_str] = pd.to_datetime(
            s[as_str].str.strip(), format="%d.%m.%Y", errors="coerce")
    if (~as_str).any():
        out.loc[~as_str] = pd.to_datetime(s[~as_str], errors="coerce")
    return out


def typed(part: pd.DataFrame):
    """Даты -> datetime, Пай/СЧА -> float, текст -> без nbsp и двойных пробелов."""
    date_col = next(c for c in part.columns if _norm(c) == "дата")
    num_cols = [c for c in part.columns if _norm(c) in {"пай", "сча"}]

    part[date_col] = to_date(part[date_col])
    for c in num_cols:
        part[c] = to_num(part[c])
    for c in part.columns:
        if c in (date_col, *num_cols) or c.startswith("_"):
            continue
        part[c] = part[c].map(
            lambda v: re.sub(r"\s+", " ", v.replace("\xa0", " ")).strip()
            if isinstance(v, str) else v)
    return part, date_col, num_cols


def period_from_name(name: str):
    """(начало, конец) из имени investfunds_20260701_20260930.xlsx или None."""
    m = _NAME_RE.match(name)
    return tuple(pd.Timestamp(g) for g in m.groups()) if m else None


# ---------------------------------------------------------------- сборка ------
files = sorted(DOWNLOAD_DIR.glob("investfunds_*"))
if not files:
    raise FileNotFoundError(
        f"в {DOWNLOAD_DIR} нет файлов investfunds_* — сначала прогоните ячейку скачивания")

frames, suspicious = [], []
for f in files:
    part, n_hdr, n_empty = extract_block(read_sheet(f), f.name)
    part, date_col, _ = typed(part)
    part["_файл"] = f.name          # след источника; снимается ниже по KEEP_SOURCE_COL

    span = "пусто"
    if not part.empty and part[date_col].notna().any():
        lo, hi = part[date_col].min(), part[date_col].max()
        span = f"{lo:{SITE_FMT}}-{hi:{SITE_FMT}}"
        want = period_from_name(f.name)
        # Если календарь на сайте не применился, файл приедет с данными чужого
        # периода. Дальше дедуп такие копии молча склеит, поэтому ловим здесь.
        if want and (lo < want[0] or hi > want[1]):
            suspicious.append(f.name)
            span += " (!) вне периода из имени"

    frames.append(part)
    print(f"{f.name}: {len(part)} строк, {span} "
          f"(шапок убрано {n_hdr}, пустых {n_empty})")

layouts = {tuple(p.columns) for p in frames}
if len(layouts) > 1:
    raise ValueError("файлы с разным набором столбцов: "
                     + "; ".join(str(sorted(l)) for l in layouts))

df = pd.concat(frames, ignore_index=True)

DATE_COL = next(c for c in df.columns if _norm(c) == "дата")
NUM_COLS = [c for c in df.columns if _norm(c) in {"пай", "сча"}]
TXT_COLS = [c for c in df.columns if c not in (DATE_COL, *NUM_COLS, "_файл")]

no_date = int(df[DATE_COL].isna().sum())      # подвальные примечания выгрузки
df = df.loc[df[DATE_COL].notna()]

was = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "_файл"])
dups = was - len(df)

df = df.sort_values([DATE_COL, *TXT_COLS], kind="stable").reset_index(drop=True)
if not KEEP_SOURCE_COL:
    df = df.drop(columns="_файл")

print(f"\nитог: {len(df)} строк, {len(df.columns)} столбцов; "
      f"без даты отброшено {no_date}, дублей снято {dups}")
if not df.empty:
    print(f"период {df[DATE_COL].min():{SITE_FMT}} - {df[DATE_COL].max():{SITE_FMT}}, "
          f"дат {df[DATE_COL].nunique()}, фондов {df[TXT_COLS[0]].nunique()}")
if suspicious:
    print("ВНИМАНИЕ: даты не применились в файлах — перекачайте их, "
          "иначе период потерян:\n  " + "\n  ".join(suspicious))
df.head()

investfunds_20210701_20210930.csv: 17042 строк, 01.07.2021-30.09.2021 (шапок убрано 0, пустых 62)
investfunds_20211001_20211231.csv: 16971 строк, 01.10.2021-30.12.2021 (шапок убрано 0, пустых 64)
investfunds_20220101_20220331.csv: 10719 строк, 09.01.2022-31.03.2022 (шапок убрано 0, пустых 65)
investfunds_20220401_20220630.csv: 12687 строк, 01.04.2022-30.06.2022 (шапок убрано 0, пустых 49)
investfunds_20220701_20220930.csv: 14932 строк, 01.07.2022-30.09.2022 (шапок убрано 0, пустых 51)
investfunds_20221001_20221231.csv: 14901 строк, 03.10.2022-30.12.2022 (шапок убрано 0, пустых 54)
investfunds_20230101_20230331.csv: 13788 строк, 09.01.2023-31.03.2023 (шапок убрано 0, пустых 57)
investfunds_20230401_20230630.csv: 15196 строк, 03.04.2023-30.06.2023 (шапок убрано 0, пустых 60)
investfunds_20230701_20230930.csv: 16359 строк, 03.07.2023-29.09.2023 (шапок убрано 0, пустых 59)
investfunds_20231001_20231231.csv: 16713 строк, 02.10.2023-29.12.2023 (шапок убрано 0, пустых 63)
investfunds_20240101

,Дата,Фонд,УК,Пай,СЧА
0,2021-07-01,А-Капитал Валютные облигации,А-Капитал,205.57,1.728619e+09
1,2021-07-01,А-Капитал Облигации,А-Капитал,342.50,1.356622e+09
2,2021-07-01,А-Капитал Сбалансированный,А-Капитал,2105.99,3.325382e+08
3,2021-07-01,АК БАРС - Акции,АК БАРС КАПИТАЛ,3856.64,4.649357e+06
4,2021-07-01,АК БАРС - Консервативный с выплатой дохода,АК БАРС КАПИТАЛ,724.27,1.412097e+08


In [224]:
df.to_excel('df_zero.xlsx')

In [225]:
df['Дата'].unique()

<DatetimeArray>
['2021-07-01 00:00:00', '2021-07-02 00:00:00', '2021-07-05 00:00:00',
 '2021-07-06 00:00:00', '2021-07-07 00:00:00', '2021-07-08 00:00:00',
 '2021-07-09 00:00:00', '2021-07-12 00:00:00', '2021-07-13 00:00:00',
 '2021-07-14 00:00:00',
 ...
 '2026-10-05 00:00:00', '2026-10-06 00:00:00', '2026-10-07 00:00:00',
 '2026-10-08 00:00:00', '2026-10-09 00:00:00', '2026-10-12 00:00:00',
 '2026-10-13 00:00:00', '2026-10-14 00:00:00', '2026-10-15 00:00:00',
 '2026-10-16 00:00:00']
Length: 1340, dtype: datetime64[ns]

In [226]:
df = df.drop_duplicates(subset=['Дата', 'Фонд'])

In [227]:
df

,Дата,Фонд,УК,Пай,СЧА
0,2021-07-01,А-Капитал Валютные облигации,А-Капитал,205.57,1.728619e+09
1,2021-07-01,А-Капитал Облигации,А-Капитал,342.50,1.356622e+09
2,2021-07-01,А-Капитал Сбалансированный,А-Капитал,2105.99,3.325382e+08
3,2021-07-01,АК БАРС - Акции,АК БАРС КАПИТАЛ,3856.64,4.649357e+06
4,2021-07-01,АК БАРС - Консервативный с выплатой дохода,АК БАРС КАПИТАЛ,724.27,1.412097e+08
...,...,...,...,...,...
367838,2026-10-16,Харизматичные идеи,Арикапитал,3739.08,6.644859e+08
367839,2026-10-16,Яндекс Пэй. Фонд акций и облигаций,ВИМ Инвестиции,104.14,8.478499e+07
367840,2026-10-16,Яндекс Пэй. Фонд денежного рынка. Рубли,ВИМ Инвестиции,114.24,4.153833e+09
367841,2026-10-16,Яндекс Пэй. Фонд золото,ВИМ Инвестиции,94.15,1.496544e+07


In [228]:
df[df.duplicated(subset=['Дата', 'Фонд'])]

,Дата,Фонд,УК,Пай,СЧА


In [268]:
assert df[df.duplicated(subset=['Дата', 'Фонд'])].shape[0] == 0, 'найдены дубликаты в данных - есть фонды с повторяющимися именами'
df[df.duplicated(subset=['Дата', 'Фонд'])]
df_1 = df.pivot(index='Дата', columns='Фонд', values='Пай').sort_index()


In [269]:
df.set_index(['Дата'])

,Фонд,УК,Пай,СЧА
Дата,,,,
2021-07-01,А-Капитал Валютные облигации,А-Капитал,205.57,1.728619e+09
2021-07-01,А-Капитал Облигации,А-Капитал,342.50,1.356622e+09
2021-07-01,А-Капитал Сбалансированный,А-Капитал,2105.99,3.325382e+08
2021-07-01,АК БАРС - Акции,АК БАРС КАПИТАЛ,3856.64,4.649357e+06
2021-07-01,АК БАРС - Консервативный с выплатой дохода,АК БАРС КАПИТАЛ,724.27,1.412097e+08
...,...,...,...,...
2026-10-16,Харизматичные идеи,Арикапитал,3739.08,6.644859e+08
2026-10-16,Яндекс Пэй. Фонд акций и облигаций,ВИМ Инвестиции,104.14,8.478499e+07
2026-10-16,Яндекс Пэй. Фонд денежного рынка. Рубли,ВИМ Инвестиции,114.24,4.153833e+09


## Приросты стоимости пая: `create_table`

Считает относительные изменения ряда на фиксированный набор периодов. Работает
от **уровней** (стоимость пая, пункты индекса), а не от накопленной доходности:
делить приросты друг на друга нельзя — `g₁₆ / g₀₉` даёт мусор, а при базе около
нуля завышает ответ в десятки раз. Правильное отношение — `P₁₆ / P₀₉ - 1`.

### Вызов

```python
create_table(series, *, kind, as_of=None, days_back=10, toler=1, pct=False)
```

| параметр | по умолчанию | смысл |
|---|---|---|
| `series` | — | `pd.Series`, индекс — даты. Одна колонка `wdf` или ряд бенчмарка |
| `kind` | **обязателен** | `'levels'` — пай/пункты; `'growth'` — доли из `series_index`. Профиль значений проверяется: перепутанный флаг даёт `ValueError`, а не тихо неверные числа |
| `as_of` | `None` | якорная дата, от которой отсчитываются все периоды. `None` → `today - days_back`. Строки только `'dd.mm.yyyy'` или `'yyyy-mm-dd'` |
| `days_back` | `10` | отступ от сегодня, когда `as_of` не задан |
| `toler` | `1` | допуск в днях: значение ищется в окне `[цель - toler, цель + toler]`, **сначала назад**, потом вперёд. `toler=0` — только точная дата |
| `pct` | `False` | `True` — проценты (2.5) вместо долей (0.025) |

Результат — `pd.Series` из 11 значений: `1W 2W 1M 3M 6M 9M 1Y 2Y 3Y YTD <год>`.
Периоды календарные (`DateOffset(months=…)`, `(years=…)`), а не кратные неделям.
`YTD` — от последнего значения прошлого года, `<год>` (например `2025`) — изменение
за весь предыдущий календарный год, от последнего значения года к последнему
значению года перед ним. Границы года берутся без `toler`: 31.12.2022 — суббота,
31.12.2023 — воскресенье, жёсткая привязка дала бы прочерк там, где данные есть.

Где значение на одном из концов не найдено — `NaN`. Прочерк рисуется при выводе:
`to_string(na_rep='—')`, `to_excel(na_rep='—')`. В данных остаётся `float64`,
иначе сломались бы сортировка по колонке и среднее по категории.

### Таблица по категории

```python
df_akcii = wdf[needed_funds['Акции']].apply(
    lambda c: create_table(c, kind='levels', as_of=ANCHOR_DATE, toler=TOLER)).T
```

`.T` обязателен: `apply` собирает периоды в индекс, а фонды в колонки.

### Что это чинит

Прежняя ячейка прогоняла паи через `series_index` и получала накопленную доходность
со своей базой у каждой колонки, после чего деление значений теряло смысл. Проверка
на «Газпромбанк - Акции»: по ценам `2918.35 / 2789.58 - 1 = +4.6161%`, через
`(1+g₁₆)/(1+g₀₉)-1` — те же `+4.6161%`, а `g₁₆/g₀₉` давало `-0.2922`.

Побочный выигрыш: `toler` закрывает риск, записанный в `docs/spec-value-growth.md` —
фонд, переставший отчитываться, не найдёт значение в окне и получит прочерки, а не
мартовские цифры в октябрьской таблице.

In [ ]:
# =============================================================================
# create_table — относительные изменения ряда на набор периодов
# =============================================================================
import datetime as _dt
import math as _math
import numbers as _numbers

import numpy as np
import pandas as pd

# Метка -> сдвиг назад от якоря. Календарные месяцы и годы, а не кратные недели:
# 1M от 17.10 это 17.09, а не 19.09, как давало timedelta(weeks=4).
PERIOD_OFFSETS = {
    "1W": pd.DateOffset(weeks=1),
    "2W": pd.DateOffset(weeks=2),
    "1M": pd.DateOffset(months=1),
    "3M": pd.DateOffset(months=3),
    "6M": pd.DateOffset(months=6),
    "9M": pd.DateOffset(months=9),
    "1Y": pd.DateOffset(years=1),
    "2Y": pd.DateOffset(years=2),
    "3Y": pd.DateOffset(years=3),
}


def _parse_day(value, name):
    """Дата -> pd.Timestamp без времени. Строки только 'dd.mm.yyyy' или 'yyyy-mm-dd'.

    pd.Timestamp('01.02.2026') на pandas 3 даёт 2 января, поэтому строки разбираем
    сами, а не доверяем конструктору.
    """
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        value = value.tz_localize(None) if value.tz is not None else value
        return value.normalize()
    if isinstance(value, np.datetime64):
        return pd.Timestamp(value).normalize()
    if isinstance(value, (_dt.datetime, _dt.date)):
        return pd.Timestamp(value).normalize()
    if isinstance(value, str):
        text = value.strip()
        for fmt in ("%d.%m.%Y", "%Y-%m-%d"):
            try:
                return pd.Timestamp(_dt.datetime.strptime(text, fmt))
            except ValueError:
                continue
        raise ValueError(f"{name}: ожидается 'dd.mm.yyyy' или 'yyyy-mm-dd', получено {value!r}")
    raise TypeError(f"{name}: ожидается строка 'dd.mm.yyyy' или дата, "
                    f"получено {type(value).__name__}")


def _as_levels(series, kind, label):
    """Ряд пользователя -> уровни float64 по дням: отсортированные, без NaN.

    kind='growth' — на входе доли (0.0 в базовой дате), наружу уходит 1 + доля:
    дальше вся арифметика одинаковая, потому что база в отношении сокращается.
    """
    if not isinstance(series, pd.Series):
        raise ValueError(f"{label}: ожидается pd.Series, получено {type(series).__name__}")

    dtype = series.dtype
    if pd.api.types.is_bool_dtype(dtype):
        raise ValueError(f"{label}: значения должны быть float/int, получен bool")
    if not (pd.api.types.is_integer_dtype(dtype) or pd.api.types.is_float_dtype(dtype)):
        for key, v in series.items():
            if v is None or v is pd.NA or (isinstance(v, float) and _math.isnan(v)):
                continue
            if isinstance(v, bool) or not isinstance(v, _numbers.Real):
                day = f"{key:%d.%m.%Y}" if isinstance(key, _dt.date) else repr(key)
                raise ValueError(f"{label}: значение не float/int: {day} -> "
                                 f"{v!r} ({type(v).__name__})")

    index = series.index
    if pd.api.types.is_datetime64_any_dtype(index.dtype):
        dates = pd.DatetimeIndex(index)
    elif index.dtype == object and len(index) and all(isinstance(x, _dt.date) for x in index):
        dates = pd.DatetimeIndex(index)
    else:
        raise ValueError(f"{label}: индекс должен быть датами, получен {index.dtype}")
    if dates.tz is not None:
        dates = dates.tz_localize(None)

    s = pd.Series(series.to_numpy(), index=dates).dropna()
    if s.empty:
        # фонд без данных не должен ронять расчёт по остальным: вернётся строка прочерков
        return pd.Series(dtype="float64", index=pd.DatetimeIndex([]))
    s = s.astype("float64").sort_index(kind="stable")
    s.index = s.index.normalize()
    s = s[~s.index.duplicated(keep="last")]  # после стабильной сортировки last = позднее время

    finite = np.isfinite(s.to_numpy())
    if not finite.all():
        bad = s[~finite]
        raise ValueError(f"{label}: значение {bad.iloc[0]} на {bad.index[0]:%d.%m.%Y}")

    if kind == "levels":
        if (s <= 0).any():
            bad = s[s <= 0]
            raise ValueError(f"{label}: kind='levels', но значение {bad.iloc[0]} на "
                             f"{bad.index[0]:%d.%m.%Y} — изменение не определено")
        return s

    if kind == "growth":
        if (s <= -1).any():
            bad = s[s <= -1]
            raise ValueError(f"{label}: kind='growth', но значение {bad.iloc[0]} на "
                             f"{bad.index[0]:%d.%m.%Y} — доля не может быть <= -1")
        if s.min() > 1:
            raise ValueError(f"{label}: kind='growth', но значения от {s.min():.2f} до "
                             f"{s.max():.2f} — похоже на уровни, а не на доли")
        return 1.0 + s

    raise ValueError(f"kind: ожидается 'levels' или 'growth', получено {kind!r}")


def _pick(levels, target, toler):
    """Значение на target или в пределах toler дней: сначала назад, потом вперёд."""
    if levels.empty or target is None:
        return np.nan
    span = pd.Timedelta(days=toler)
    back = levels.loc[target - span:target]      # target включительно: точная дата выигрывает
    if len(back):
        return back.iloc[-1]                     # самая поздняя дата назад
    if toler:
        fwd = levels.loc[target + pd.Timedelta(days=1):target + span]
        if len(fwd):
            return fwd.iloc[0]                   # самая ранняя дата вперёд
    return np.nan


def _year_end(levels, year):
    """Последнее значение календарного года или NaN.

    Граница года ищется без toler: 31.12.2022 — суббота, 31.12.2023 — воскресенье,
    и жёсткая привязка к 31.12 дала бы прочерк там, где данные есть.
    """
    if levels.empty:
        return np.nan
    chunk = levels[levels.index.year == year]
    return chunk.iloc[-1] if len(chunk) else np.nan


def _ratio(num, den):
    """num / den - 1 или NaN, если какой-то конец не найден."""
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return num / den - 1


def create_table(series, *, kind, as_of=None, days_back=10, toler=1, pct=False):
    """Относительные изменения ряда на набор периодов -> pd.Series.

    series    — pd.Series с датами в индексе: стоимость пая, пункты индекса
                (kind='levels') или накопленная доходность в долях (kind='growth')
    kind      — обязателен: 'levels' или 'growth'. Профиль значений проверяется,
                перепутанный флаг даёт ValueError, а не тихо неверные числа
    as_of     — якорная дата, от которой отсчитываются все периоды;
                None — today - days_back
    days_back — на сколько дней отступить от сегодня, когда as_of не задан
    toler     — допуск в днях: значение ищется в окне [цель - toler, цель + toler],
                сначала назад, потом вперёд. toler=0 — только точная дата
    pct       — True: проценты (2.5) вместо долей (0.025)

    Индекс результата: 1W, 2W, 1M, 3M, 6M, 9M, 1Y, 2Y, 3Y, YTD и год (например
    '2025') — изменение за весь предыдущий календарный год. Где значение на одном
    из концов не найдено — NaN (прочерк рисуется при выводе, na_rep='—').

    Таблица по категории фондов:
        wdf[funds].apply(lambda c: create_table(c, kind='levels')).T
    .T обязателен: apply собирает периоды в индекс, а фонды в колонки.
    """
    if isinstance(toler, bool) or not isinstance(toler, _numbers.Integral) or toler < 0:
        raise ValueError(f"toler: ожидается целое >= 0, получено {toler!r}")
    if isinstance(days_back, bool) or not isinstance(days_back, _numbers.Integral):
        raise ValueError(f"days_back: ожидается целое, получено {days_back!r}")

    name = series.name if isinstance(series, pd.Series) else None
    label = str(name) if name is not None else "ряд"
    levels = _as_levels(series, kind, label)

    anchor = _parse_day(as_of, "as_of")
    if anchor is None:
        anchor = pd.Timestamp(_dt.date.today() - _dt.timedelta(days=int(days_back)))

    now = _pick(levels, anchor, toler)
    out = {label_: _ratio(now, _pick(levels, (anchor - off).normalize(), toler))
           for label_, off in PERIOD_OFFSETS.items()}

    prev_year = anchor.year - 1
    prev_end = _year_end(levels, prev_year)
    out["YTD"] = _ratio(now, prev_end)
    out[str(prev_year)] = _ratio(prev_end, _year_end(levels, prev_year - 1))

    res = pd.Series(out, dtype="float64", name=name)
    res.index.name = "период"
    return res * 100 if pct else res

In [ ]:
# df_1 = pd.read_excel('df_1.xlsx')
df_1.to_excel('df_1_excel.xlsx')

TOLER = 1   # допуск в днях при поиске значения на целевую дату (0 — только точная дата)

# Окно выгрузки считаем от самого длинного периода в PERIOD_OFFSETS, а не от 3*365:
# иначе при добавлении, скажем, 5Y данные молча обрежутся и колонка станет прочерком.
# Плюс год назад на границу «за весь предыдущий год» и 30 дней запаса на toler и праздники.
_oldest_target = min(pd.Timestamp(ANCHOR_DATE) - off for off in PERIOD_OFFSETS.values())
start_date = min(_oldest_target, pd.Timestamp(ANCHOR_DATE.year - 2, 1, 1)) - pd.Timedelta(days=30)
finish_date = pd.Timestamp(ANCHOR_DATE)

mask = (df_1.index >= start_date) & (df_1.index <= finish_date)
wdf = df_1[mask]
print(f'окно {start_date:%d.%m.%Y} - {finish_date:%d.%m.%Y}, '
      f'якорь {ANCHOR_DATE:%d.%m.%Y}, toler={TOLER}, строк {len(wdf)}')

In [232]:
wdf.to_excel('wdf_1.xlsx')

In [288]:
needed_funds = {
    "Акции" :
    ['Газпромбанк - Акции',
    'РСХБ - Фонд Акций',
    'Альфа-Капитал Ликвидные акции',
    'Просто. Акции', # промсвязь
    'ВИМ – Акции. Российские эмитенты',
    'Первая - Фонд российских акций',
    'Райффайзен - Акции',
    'Апрель Капитал - Акции',
    'Атон - Петр Столыпин']
    ,
    "Смешанные" : 
    ['ВИМ - Сбалансированный. Консервативный',
    'Газпромбанк - Сбалансированный',
    'РСХБ - Фонд Сбалансированный',
    'ВИМ - Сбалансированные инвестиции',
    'Просто. Баланс', # промсвязь
    'ВИМ - Сбалансированный',
    'ВИМ - Накопительный резерв',
    'Первая - Сбалансированный',
    'Апрель Капитал - Сбалансированный',
    'Альфа-Капитал Баланс']
    ,
    "Облигации" :
    ['Т-Капитал Облигации',
    'Просто. Облигации', # промсвязь
    'Газпромбанк - Облигации плюс',
    'ВИМ - Казначейский',
    'ВИМ - Облигации. Российские эмитенты',
    'РСХБ - Фонд Облигаций',
    'Альфа-Капитал Облигации Плюс',
    'Первая - Фонд Рублевые сбережения',
    'Райффайзен - Облигации',
    'Атон - Фонд облигаций']
    ,
    "Валютные облигации" :
    ['Ингосстрах - валютные облигации',
    'Т-Капитал Локальные валютные облигации',
    'Курс на Восток',
    'ВИМ - Новые валютные облигации',
    'Первая – Фонд Валютных Облигаций',
    'БКС Российские Еврооблигации']
}

flat_funds_list = [fund for sublist in needed_funds.values() for fund in sublist]

# Фильтруем DataFrame по плоскому списку
wdf = wdf[flat_funds_list]

assert wdf.shape[1] == len(flat_funds_list), 'Количество фондов в списке фондов "needed_funds" не совпадает с количеством в dataframe "wdf"'

In [289]:
wdf

Фонд,Газпромбанк - Акции,РСХБ - Фонд Акций,Альфа-Капитал Ликвидные акции,Просто. Акции,ВИМ – Акции. Российские эмитенты,Первая - Фонд российских акций,Райффайзен - Акции,Апрель Капитал - Акции,Атон - Петр Столыпин,ВИМ - Сбалансированный. Консервативный,...,Альфа-Капитал Облигации Плюс,Первая - Фонд Рублевые сбережения,Райффайзен - Облигации,Атон - Фонд облигаций,Ингосстрах - валютные облигации,Т-Капитал Локальные валютные облигации,Курс на Восток,ВИМ - Новые валютные облигации,Первая – Фонд Валютных Облигаций,БКС Российские Еврооблигации
Дата,,,,,,,,,,,,,,,,,,,,,
2023-10-13,2889.23,4373.57,7648.15,10820.97,1426.47,16447.73,52237.36,1170.23,4127.80,40.56,...,5196.40,43294.90,27824.78,2942.03,NaN,NaN,1340.22,1024.05,NaN,12.395935
2023-10-16,2927.52,4434.20,7728.24,10997.43,1445.72,16673.27,52975.87,1187.33,4183.25,40.81,...,5210.29,43420.11,27865.46,2946.43,NaN,NaN,1339.00,1026.99,NaN,12.426168
2023-10-17,2944.59,4448.62,7755.75,11049.48,1453.22,16712.13,53128.73,1193.83,4205.16,40.88,...,5209.59,43471.10,27836.72,2948.16,2211.05,NaN,1343.02,1028.22,NaN,12.442943
2023-10-18,2945.81,4467.93,7806.48,11123.18,1463.84,16776.12,53395.47,1199.94,4216.02,40.90,...,5202.42,43443.14,27830.07,2940.38,2205.54,NaN,1341.84,1029.53,NaN,12.460262
2023-10-19,2954.40,4480.38,7842.71,11114.41,1468.09,16856.57,53805.35,1202.85,4226.27,40.88,...,5191.59,43368.94,27806.36,2939.92,2199.13,NaN,1341.74,1030.38,NaN,12.473404
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-10-12,2869.63,4138.35,7737.70,10211.38,1624.46,16408.40,48722.95,1163.44,3774.32,57.01,...,7202.54,64487.39,36491.77,4092.51,2686.27,11.2244,1383.19,1097.43,1460.20,12.620000
2026-10-13,2916.25,4190.28,7855.44,10353.31,1658.46,16630.55,49511.06,1183.92,3849.67,57.27,...,7215.83,64573.28,36559.09,4098.51,2686.43,11.1765,1376.33,1097.96,1455.29,12.720000
2026-10-14,2912.76,4187.75,7866.42,10350.96,1648.83,16620.75,49442.45,1173.99,3832.29,57.26,...,7230.84,64726.49,36665.72,4113.81,2676.79,11.1353,1373.20,1096.01,1457.09,12.650000


In [290]:
wdf[750:]

Фонд,Газпромбанк - Акции,РСХБ - Фонд Акций,Альфа-Капитал Ликвидные акции,Просто. Акции,ВИМ – Акции. Российские эмитенты,Первая - Фонд российских акций,Райффайзен - Акции,Апрель Капитал - Акции,Атон - Петр Столыпин,ВИМ - Сбалансированный. Консервативный,...,Альфа-Капитал Облигации Плюс,Первая - Фонд Рублевые сбережения,Райффайзен - Облигации,Атон - Фонд облигаций,Ингосстрах - валютные облигации,Т-Капитал Локальные валютные облигации,Курс на Восток,ВИМ - Новые валютные облигации,Первая – Фонд Валютных Облигаций,БКС Российские Еврооблигации
Дата,,,,,,,,,,,,,,,,,,,,,
2026-09-24,2762.09,4016.76,7482.34,9898.76,1568.87,15743.37,46903.58,1114.45,3652.61,56.55,...,7263.37,65127.40,36663.85,4132.09,2606.70,11.2161,1384.80,1102.52,1466.37,12.89
2026-09-25,2719.65,3959.39,7368.04,9748.09,1547.25,15559.25,46190.96,1100.28,3590.55,56.42,...,7255.24,65081.09,36593.82,4124.53,2596.32,11.2025,1386.21,1096.89,1460.49,12.73
2026-09-28,2703.47,3929.84,7329.10,9679.36,1536.00,15465.99,45969.52,1085.59,3553.48,56.24,...,7233.67,64810.05,36460.32,4099.86,2600.13,11.2084,1383.56,1099.75,1462.28,12.83
2026-09-29,2705.70,3947.16,7352.16,9715.10,1539.77,15524.45,46183.55,1087.70,3573.07,56.26,...,7202.90,64457.01,36356.78,4085.83,2588.17,11.1735,1381.83,1095.57,1459.72,12.72
2026-09-30,2704.71,3957.83,7372.73,9745.72,1541.86,15577.43,46377.11,1089.93,3583.53,56.26,...,7203.41,64498.71,36353.68,4090.26,2572.02,11.1327,1382.45,1085.72,1447.79,12.50
2026-10-01,2717.58,3983.37,7429.86,9821.26,1552.29,15656.43,46740.97,1093.44,3611.67,56.36,...,7202.53,64532.69,36356.80,4089.74,2665.77,11.1044,1374.39,1089.65,1447.34,12.68
2026-10-02,2719.47,3983.21,7430.42,9833.13,1554.99,15679.96,46765.03,1094.74,3611.47,56.38,...,7204.77,64472.97,36379.52,4089.07,2678.51,11.1333,1374.74,1095.81,1451.16,12.84
2026-10-05,2772.71,4053.26,7548.31,9981.01,1582.74,15897.22,47508.03,1112.92,3676.73,56.62,...,7214.90,64596.19,36441.47,4096.47,2720.15,11.2891,1385.35,1115.06,1474.71,13.20
2026-10-06,2788.13,4064.35,7582.50,10021.10,1587.19,15928.31,47674.84,1112.43,3682.65,56.67,...,7205.75,64505.20,36435.06,4085.81,2723.46,11.3430,1398.75,1115.76,1488.78,13.01


In [ ]:
# Четыре таблицы: индекс — фонды, колонки — периоды. Паи подаём как есть, уровнями:
# прогонять их через series_index нельзя, деление накопленных доходностей даёт мусор.
tables = {
    name: wdf[funds].apply(
        lambda col: create_table(col, kind='levels', as_of=ANCHOR_DATE, toler=TOLER)).T
    for name, funds in needed_funds.items()
}

df_akcii = tables['Акции']
df_mixed = tables['Смешанные']
df_bonds = tables['Облигации']
df_fxbonds = tables['Валютные облигации']

# Колонка за предыдущий год от якоря не зависит, поэтому «нет данных на якорь»
# проверяем только по периодам, которые от него считаются.
_anchor_cols = [c for c in next(iter(tables.values())).columns if c != str(ANCHOR_DATE.year - 1)]
for name, t in tables.items():
    stale = t[_anchor_cols].isna().all(axis=1)
    print(f'{name}: {t.shape[0]} фондов x {t.shape[1]} периодов'
          + (f'; нет данных на якорь: {", ".join(t.index[stale])}' if stale.any() else ''))

In [293]:
wdf[730:]

Фонд,Газпромбанк - Акции,РСХБ - Фонд Акций,Альфа-Капитал Ликвидные акции,Просто. Акции,ВИМ – Акции. Российские эмитенты,Первая - Фонд российских акций,Райффайзен - Акции,Апрель Капитал - Акции,Атон - Петр Столыпин,ВИМ - Сбалансированный. Консервативный,...,Альфа-Капитал Облигации Плюс,Первая - Фонд Рублевые сбережения,Райффайзен - Облигации,Атон - Фонд облигаций,Ингосстрах - валютные облигации,Т-Капитал Локальные валютные облигации,Курс на Восток,ВИМ - Новые валютные облигации,Первая – Фонд Валютных Облигаций,БКС Российские Еврооблигации
date,,,,,,,,,,,,,,,,,,,,,
2026-09-25,-0.058694,-0.094701,-0.036625,-0.099148,0.084671,-0.054018,-0.115749,-0.059775,-0.130154,0.391026,...,0.396205,0.503205,0.315152,0.401933,0.174248,0.120295,0.034315,0.071129,0.119467,0.026950
2026-09-28,-0.064294,-0.101457,-0.041716,-0.105500,0.076784,-0.059688,-0.119988,-0.072328,-0.139135,0.386588,...,0.392054,0.496944,0.310354,0.393548,0.175971,0.120885,0.032338,0.073922,0.120839,0.035017
2026-09-29,-0.063522,-0.097497,-0.038701,-0.102197,0.079427,-0.056134,-0.115890,-0.070525,-0.134389,0.387081,...,0.386133,0.488790,0.306633,0.388779,0.170561,0.117395,0.031047,0.069840,0.118877,0.026143
2026-09-30,-0.063865,-0.095057,-0.036011,-0.099367,0.080892,-0.052913,-0.112185,-0.068619,-0.131855,0.387081,...,0.386231,0.489753,0.306522,0.390285,0.163257,0.113315,0.031510,0.060222,0.109732,0.008395
2026-10-01,-0.059410,-0.089218,-0.028542,-0.092386,0.088204,-0.048110,-0.105220,-0.065620,-0.125038,0.389546,...,0.386062,0.490538,0.306634,0.390108,0.205658,0.110484,0.025496,0.064059,0.109387,0.022916
2026-10-02,-0.058756,-0.089254,-0.028468,-0.091289,0.090097,-0.046679,-0.104759,-0.064509,-0.125086,0.390039,...,0.386493,0.489159,0.307450,0.389880,0.211420,0.113375,0.025757,0.070075,0.112315,0.035823
2026-10-05,-0.040329,-0.073238,-0.013054,-0.077623,0.109550,-0.033470,-0.090535,-0.048973,-0.109276,0.395957,...,0.388442,0.492005,0.309677,0.392396,0.230253,0.128955,0.033674,0.088873,0.130366,0.064865
2026-10-06,-0.034992,-0.070702,-0.008584,-0.073919,0.112670,-0.031580,-0.087342,-0.049392,-0.107842,0.397189,...,0.386681,0.489903,0.309446,0.388772,0.231750,0.134345,0.043672,0.089556,0.141151,0.049538
2026-10-07,-0.029935,-0.068130,-0.005501,-0.070470,0.118138,-0.027613,-0.082702,-0.046008,-0.103428,0.398915,...,0.386573,0.490850,0.309350,0.389500,0.236435,0.138736,0.047015,0.091304,0.143060,0.044697


In [264]:
con = pyodbc.connect(
    "driver={SQL Server};"
    r"server=AMDBRATES\DBMVTBAM01;"
    "database=Rates;"
    "trusted_connection=yes"
)
def fetch_micex_index(security_id, start, finish):
    df = pd.read_sql(
        f""" 
        select TRADEDATE, [CLOSE] as [{security_id}] from rates.dbo.dbo_MICEX_Index
        where securityid = ? and tradedate >= CONVERT(date, ?, 104) and tradedate <= CONVERT(date, ?, 104)
        order by TRADEDATE
        """,
        con=con,
        params=[security_id, start, finish],
    )
    return df.set_index("TRADEDATE").squeeze()



In [ ]:
# Проверка флага kind: уровни и приросты обязаны дать одну и ту же таблицу,
# потому что база в отношении P(t0)/P(t1) сокращается.
mcftr = fetch_micex_index('MCFTR', start_date, finish_date)

_lv = create_table(mcftr, kind='levels', as_of=ANCHOR_DATE, toler=TOLER)
_gr = create_table(mcls.series_index([(mcftr, 1)], name='MCFTR'),
                   kind='growth', as_of=ANCHOR_DATE, toler=TOLER)
pd.concat({'levels': _lv, 'growth': _gr, 'разница': (_lv - _gr).abs()}, axis=1)

In [ ]:
# Бенчмарк добавляется строкой, а не колонкой: в таблице теперь фонды x периоды.
# Через series_index — он же считает и взвешенные композиты, если бенчмарк составной.
df_akcii.loc['MCFTR'] = create_table(mcls.series_index([(mcftr, 1)], name='MCFTR'),
                                     kind='growth', as_of=ANCHOR_DATE, toler=TOLER)

# Остальные бенчмарки — так же, каждый в свою таблицу:
# df_bonds.loc['RGBITR'] = create_table(
#     mcls.series_index([(fetch_micex_index('RGBITR', start_date, finish_date), 1)],
#                       name='RGBITR'), kind='growth', as_of=ANCHOR_DATE, toler=TOLER)

In [ ]:
# Вывод: проценты, два знака, прочерк вместо NaN. В данных остаётся float64.
for name, t in (('Акции', df_akcii), ('Смешанные', df_mixed),
                ('Облигации', df_bonds), ('Валютные облигации', df_fxbonds)):
    print(f'\n=== {name} ===')
    print(t.mul(100).round(2).to_string(na_rep='—'))

# в Excel одной книгой, по листу на категорию
# with pd.ExcelWriter(f'perform_{ANCHOR_DATE:%d%m%Y}.xlsx') as xl:
#     for name, t in tables.items():
#         t.mul(100).round(2).to_excel(xl, sheet_name=name[:31], na_rep='—')